# 02 | Data Quality Assessment

**Project:** Dubai Short-Term Rental Intelligence  |  **Source:** AirROI Dubai (Kaggle download)  |  **Reporting coverage:** March 2021–February 2026

**Business question:** What data risks could make a pricing comparison or Power BI KPI misleading? The notebook produces auditable quality checks and prioritizes issues before any cleaning.

**Interpretation:** Figures describe the supplied AirROI sample. Occupancy and revenue are provider estimates, not audited bookings or market-wide totals. The source is independent of Airbnb.

## Reproducibility and dependencies

Run from top to bottom in Jupyter using the Python 3 kernel. The notebook resolves its project root automatically when Jupyter starts in the repository root or `notebooks/`. Source files belong in `data/raw/` (excluded from GitHub because redistribution rights have not been established). Regenerated files are written to `data/processed/`, `powerbi/data/` or `reports/`, never into the raw directory. The full workflow and dependencies are documented in the project README.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CWD, CWD.parent, CWD.parent.parent]
                     if (p / 'README.md').exists() and (p / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run Jupyter from the project root or notebooks/ directory.')
RAW = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
print('Project:', PROJECT_ROOT.name)

## Quality framework

Checks cover listing-month keys, source overlap, month length, AED currency, coordinate ranges, prices, source estimates, and sample coverage. A failed check is reported rather than silently corrected.

In [2]:
a=pd.read_csv(RAW/'past_rates.csv',dtype={'listing_id':'string'},low_memory=False)
b=pd.read_csv(RAW/'listings_monthly.csv',dtype={'listing_id':'string','host_id':'string'},low_memory=False)
for df in (a,b): df['month_date']=pd.to_datetime(df['month_date'],errors='coerce')
checks=[]
def check(test,failed,total,severity,comment):
    checks.append({'check':test,'issues':int(failed),'records_tested':int(total),
                   'severity':severity,'details':comment})
for name,df in [('historical',a),('enriched',b)]:
    check(f'{name}: null listing or month', df.listing_id.isna().sum()+df.month_date.isna().sum(),len(df),'critical','Month and listing identify each observation')
    check(f'{name}: duplicate listing-month',df.duplicated(['listing_id','month_date']).sum(),len(df),'critical','One record per listing per calendar month')
    check(f'{name}: non-AED currency',df.currency.ne('AED').sum(),len(df),'critical','Only native monetary columns treated as AED')
    check(f'{name}: invalid occupancy',(~df.occupancy.between(0,1)).sum(),len(df),'critical','Expected modeled share between 0 and 1')
    check(f'{name}: invalid reported rate',((df.native_avg_daily_rate<0)|df.native_avg_daily_rate.isna()).sum(),len(df),'major','Zero may be meaningful, negatives are not')
    check(f'{name}: bad month-day total',((df.available_days+df.unavailable_days)!=df.month_date.dt.days_in_month).sum(),len(df),'major','Available + unavailable should match calendar days')
    for metric in ('native_revenue', 'native_revpar'):
        check(f'{name}: invalid {metric}',((df[metric]<0)|df[metric].isna()).sum(),len(df),'major','Required modeled monetary measure must be present and nonnegative')
    coords_invalid=(df.latitude.isna()|df.longitude.isna()|~df.latitude.between(-90,90)|~df.longitude.between(-180,180))
    check(f'{name}: invalid global coordinate ranges',coords_invalid.sum(),len(df),'major','Missing or physically invalid geographic coordinates')
    # Broad flagging envelope, not a substitute for geocoding a property to Dubai.
    outside_dubai_envelope=(~df.latitude.between(24,26)|~df.longitude.between(54,57))
    check(f'{name}: outside broad Dubai coordinates envelope',outside_dubai_envelope.sum(),len(df),'warning','Investigate locations outside broad regional bounds; this is not proof of geocoding accuracy')
    check(f'{name}: missing neighborhood',df.neighborhood.isna().sum(),len(df),'warning','Avoid unverified neighborhood conclusions')
    check(f'{name}: missing bedrooms',df.bedrooms.isna().sum(),len(df),'warning','Limits eligible peer groups')
quality=pd.DataFrame(checks)
display(quality)
print('Critical issues:',int(quality.loc[quality.severity=='critical','issues'].sum()))

## Enriched-file consistency and scope

We compare shared values on matching listing-month keys and explicitly verify that the enrichment does not introduce additional observations.

In [3]:
fields=['active','native_avg_daily_rate','native_revenue','native_revpar','occupancy','bedrooms','latitude','longitude','neighborhood']
shared=a[['listing_id','month_date']+fields].merge(b[['listing_id','month_date']+fields],on=['listing_id','month_date'],suffixes=('_main','_extra'),validate='one_to_one')
res=[]
for field in fields:
    x,y=shared[field+'_main'],shared[field+'_extra']
    mismatch=~(x.eq(y)|(x.isna()&y.isna()))
    res.append({'field':field,'compared':len(shared),'disagreements':int(mismatch.sum())})
consistency=pd.DataFrame(res)
display(consistency)
unknown=b[['listing_id','month_date']].merge(a[['listing_id','month_date']],how='left',indicator=True,on=['listing_id','month_date'])
print('Enriched records without history:',int(unknown['_merge'].eq('left_only').sum()))

## Coverage volatility and missingness in the latest snapshot

The report's baseline month is the latest one supplied. Missing-neighborhood and zero-estimate rates are documented rather than imputed.

In [4]:
import matplotlib.pyplot as plt
monthly=(a.groupby('month_date',as_index=False).agg(observed=('listing_id','nunique'),active=('active','sum'))
         .sort_values('month_date'))
monthly['change_pct']=monthly.observed.pct_change(fill_method=None)*100
latest=a.month_date.max();l=a[a.month_date==latest];l_active=l[l.active]
audit=pd.DataFrame([
    {'indicator':'Neighborhood missing','affected':int(l.neighborhood.isna().sum()),'denominator':len(l)},
    {'indicator':'Bedrooms missing','affected':int(l.bedrooms.isna().sum()),'denominator':len(l)},
    {'indicator':'Active with zero estimated occupancy','affected':int((l_active.occupancy==0).sum()),'denominator':len(l_active)},
])
audit['pct']=audit.affected/audit.denominator*100
display(audit)
fig,ax=plt.subplots(figsize=(8,3.6))
ax.barh(audit.indicator,audit.pct)
ax.set(title=f'Quality risks | {latest:%b %Y}',xlabel='Share of relevant sample (%)',xlim=(0,100))
fig.tight_layout()
(REPORTS / 'figures').mkdir(parents=True, exist_ok=True)
fig.savefig(REPORTS / 'figures' / '02_quality_risks.png', dpi=140)
plt.show()
print('Periods with >25% absolute sample-count change:',int((monthly.change_pct.abs()>25).sum()))

## Export reproducible audit tables

Exports are derived summaries only. Original files remain unchanged. Warnings become documented dashboard caveats, not reasons to fabricate missing information.

In [5]:
dest=REPORTS/'data_quality_assessment';dest.mkdir(parents=True,exist_ok=True)
quality.to_csv(dest/'quality_rules.csv',index=False)
consistency.to_csv(dest/'shared_field_agreement.csv',index=False)
audit.to_csv(dest/'latest_snapshot_risks.csv',index=False)
monthly.to_csv(dest/'sample_coverage.csv',index=False)
assert int(quality.loc[quality.severity.isin(['critical','major']),'issues'].sum())==0, 'Critical or major source validation failed.'
print('Passed critical and major checks; warnings document remaining coverage limitations.')
assert (consistency.disagreements==0).all(), 'Shared source values diverge.'
print('Validated; audit exports:',dest.relative_to(PROJECT_ROOT))

## Key findings and analytical implications

The assessment found **no critical or major integrity violations in the tested rules** for the two source CSVs. Both datasets maintain a unique **listing–month** key, and the **22,629 overlapping listing–month observations** have no disagreements across the nine compared fields. The enriched extract is a subset of the historical file, not an additional population of listings.

### Material data-quality findings

| Quality finding | Observed result | Implication for analysis |
|:--|--:|:--|
| Critical or major integrity issues in the checks above | **0** | The tested structural rules passed; this does not guarantee that every source value is accurate. |
| Disagreements across shared fields | **0 in each compared field across 22,629 overlapping rows** | The enriched data can supplement the historical data without double-counting shared listing–month records. |
| Missing neighborhood in the latest snapshot | **1,919 / 2,471 (77.66%)** | Neighborhood-level rankings are not defensible without independently validated location mapping. |
| Missing bedrooms in the latest snapshot | **298 / 2,471 (12.06%)** | Listings without bedroom counts cannot enter bedroom-matched peer comparisons without an explicit alternate rule. |
| Active listings with zero provider-estimated occupancy | **919 / 1,955 (47.01%)** | Estimated occupancy requires careful interpretation; a zero estimate is not direct evidence of zero confirmed bookings. |
| Monthly sample-count changes above 25% in absolute value | **5 periods** | Raw changes in listing counts may reflect source coverage shifts rather than actual market growth or decline. |

*Rates above use the latest observed month (February 2026), unless otherwise noted. Denominators differ by metric: missingness uses all 2,471 observed listings; zero estimated occupancy uses 1,955 active listings.*

### Decisions for the analytical pipeline

**Data preparation.** Preserve the original files and listing–month grain. Use the historical CSV as the primary observation table; incorporate supplemental attributes only when they contribute distinct fields. Record exclusions and null handling explicitly instead of silently imputing location, bedrooms, or estimated performance.

**Competitor benchmarking.** Use validated coordinates, room type, and bedroom count where available to form plausible comparison groups. Report peer counts and matching criteria. Label listings without sufficient comparable observations as *insufficient evidence*, rather than assigning a misleading benchmark.

**Business intelligence.** Label occupancy and revenue as **AirROI/provider estimates**. Display the reporting period, sample coverage, and missing-data limitations alongside key Power BI measures. Avoid claiming that observations represent the entire Dubai market, or that unavailable calendar nights necessarily indicate bookings.

**Next notebook — `03_data_preparation.ipynb`.** Build documented analytical tables and retain a reproducible record of transformations and excluded observations.